# 01. Первый агент: state, tools и filesystem

## Что агент пока не умеет

После `00` у нас есть идея agent harness, но ещё нет агента, который сам исследует рабочую директорию.

> Модель умеет рассуждать о коде только тогда, когда человек принёс код в контекст. Агент сам исследует окружение и решает, какие данные ему нужны.


## Какую способность добавляем

Сначала создаём минимальный Deep Agent без filesystem. Затем добавляем `FilesystemBackend` и показываем контраст до/после.

#TODO: Перегружено, нужна диаграмма!
```text
Model  → решает, что делать дальше #TODO - Варианты подключения
State  → хранит сообщения и результаты действий #TODO - про все подробнее
Tools  → позволяют воздействовать на окружение
```


## Workspace boundary

В нашем demo workspace — корень репозитория. `virtual_mode=True` нормализует пути внутри backend, но это не полноценная OS-песочница.

```text
Физический путь: локальный каталог на компьютере
Виртуальный путь: рабочее пространство агента внутри backend
```

Shell выключен по умолчанию. Если включить `OPENCLAW_ENABLE_LOCAL_SHELL=1`, shell не наследует env процесса (`inherit_env=False`). + #TODO: Картинка! #Объяснить что такое virtual mode и что такое b
ackend и как используется


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, model_name, print_stage_context, register_graphs, workspace_root, write_text

print_stage_context()
print("pyproject:", REPO_ROOT / "pyproject.toml")


In [ ]:
from deepagents import create_deep_agent

BASE_PROMPT = """\
You are SberClaw at stage 01. Respond in Russian by default.
Explain honestly whether you can inspect repository files.
"""

minimal_agent = create_deep_agent(model=model_name(), tools=[], system_prompt=BASE_PROMPT)
print(type(minimal_agent).__name__)


In [ ]:
from textwrap import dedent

ENTRYPOINT = dedent('''\
from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv


DEFAULT_MODEL = "openrouter:tencent/hy3:free"


def _find_repo_root(start: Path | None = None) -> Path:
    root = (start or Path.cwd()).resolve()
    while root != root.parent and not (root / "pyproject.toml").exists():
        root = root.parent
    if not (root / "pyproject.toml").exists():
        raise RuntimeError("Could not find repository root with pyproject.toml")
    return root


REPO_ROOT = _find_repo_root()
load_dotenv(REPO_ROOT / ".env")


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", str(REPO_ROOT))).expanduser().resolve()


def _backend():
    root = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") == "1":
        from deepagents.backends import LocalShellBackend

        return LocalShellBackend(
            root_dir=root,
            virtual_mode=True,
            inherit_env=False,
            timeout=120,
            max_output_bytes=80_000,
        )

    from deepagents.backends import FilesystemBackend

    return FilesystemBackend(root_dir=root, virtual_mode=True)


MINIMAL_PROMPT = """You are OpenClaw at stage 01 before the real workspace backend is attached.
Respond in the user's language; default to Russian.
If the user asks about repository files, explain that this graph does not have access to the real repository workspace yet.
Do not guess repository structure from the prompt.
"""

FILESYSTEM_PROMPT = """You are OpenClaw at stage 01 with the real workspace backend attached.
Respond in the user's language; default to Russian.
You can inspect the repository through the filesystem tools: ls, glob, grep, and read_file.
For repository claims, first call filesystem tools and cite the files you inspected.
Do not say that the backend is disabled; this graph is the filesystem-enabled variant.
"""


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=[],
    system_prompt=MINIMAL_PROMPT,
)

agent_with_filesystem = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=[],
    system_prompt=FILESYSTEM_PROMPT,
    backend=_backend(),
)
''')

entrypoint = write_text("agents/openclaw_01_agent_and_filesystem.py", ENTRYPOINT)
config_path = register_graphs({
    "openclaw_01": "./agents/openclaw_01_agent_and_filesystem.py:agent",
    "openclaw_01_fs": "./agents/openclaw_01_agent_and_filesystem.py:agent_with_filesystem",
})

print("Entrypoint:", entrypoint.relative_to(REPO_ROOT))
print("LangGraph config:", config_path.relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Найди файл pyproject.toml и назови имя проекта и три зависимости. Если у тебя нет доступа к файлам, прямо скажи об этом.
```

### Ожидаемое поведение

1. `openclaw_01` честно говорит, что не имеет файлового доступа.
2. `openclaw_01_fs` читает `pyproject.toml` и цитирует вывод из файла.

### Что изменилось относительно предыдущего этапа

Появился контраст между моделью без среды и агентом с workspace.

### Текущее ограничение

Агент умеет исследовать локальную среду, но пока не взаимодействует с системами за пределами репозитория.
